In [1]:
import gc
import re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# 0. 공통 설정
#    v3 변경사항:
#    - 추론 입력을 학습과 동일한 전처리로 고정
#    - 단일 모델 제출 + 다중 모델 평균 logit 앙상블 지원
#    - K-Fold 결과 폴더(results_dktc_kfold_* )를 바로 제출에 사용할 수 있도록 지원
# ============================================================
CONFIG = {
    "test_path": Path("./data/real/test.csv"),
    "sample_submission_path": Path("./data/real/submission.csv"),
    "output_dir": Path("./submission"),
    "batch_size": 32,
    "max_length": 512,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

LABEL_NAMES = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]
NUM_LABELS = len(LABEL_NAMES)
TURN_TOKEN = "[턴]"


def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 1. 학습 노트북과 동일한 텍스트 전처리
#    - merge_preprocessing_pipeline.ipynb 기준
# ============================================================
def clean_text(text: str) -> str:
    """한국어 대화체 텍스트 정제"""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                  # 반복문자 축소
    text = re.sub(r"([ㄱ-ㅎㅏ-ㅣ])\1{2,}", r"\1\1", text)        # 자모 반복 축소
    text = re.sub(r"[^\w\s가-힣a-zA-Z0-9.,!?~\n]", " ", text)   # 특수문자 제거
    text = re.sub(r"[ \t]+", " ", text)                          # 연속 공백 축소
    text = re.sub(r"\n+", "\n", text)                           # 연속 줄바꿈 축소
    return text.strip()


def normalize_conversation(text: str) -> str:
    """대화 턴 구분을 [턴] 토큰으로 변환"""
    if not isinstance(text, str):
        return ""
    turns = [t.strip() for t in text.split("\n") if t.strip()]
    return f" {TURN_TOKEN} ".join(turns)


def build_inference_texts(conversation_series: pd.Series) -> list[str]:
    """test.csv raw conversation -> 학습과 동일한 text 입력으로 변환"""
    conversation_clean = conversation_series.apply(clean_text)
    conversation_norm = conversation_clean.apply(normalize_conversation)

    empty_mask = conversation_norm.str.len().eq(0)
    if empty_mask.any():
        empty_count = int(empty_mask.sum())
        raise ValueError(f"전처리 후 빈 문자열이 발생했습니다: {empty_count}개")

    return conversation_norm.tolist()


# ============================================================
# 2. 입력 검증
# ============================================================
def detect_submission_columns(sample_df: pd.DataFrame):
    cols = sample_df.columns.tolist()

    id_candidates = ["file_name", "idx", "id"]
    target_candidates = ["class", "target", "label"]

    id_col = next((c for c in id_candidates if c in cols), None)
    target_col = next((c for c in target_candidates if c in cols), None)

    if id_col is None or target_col is None:
        raise ValueError(f"sample submission 컬럼 인식 실패: {cols}")

    return id_col, target_col


def validate_inputs(test_df: pd.DataFrame, sample_df: pd.DataFrame):
    if "conversation" not in test_df.columns:
        raise ValueError("test.csv에 conversation 컬럼이 없습니다.")

    if test_df["conversation"].isna().any():
        na_count = int(test_df["conversation"].isna().sum())
        raise ValueError(f"test.csv conversation 컬럼에 결측치가 있습니다: {na_count}개")

    if len(test_df) != len(sample_df):
        raise ValueError(
            f"행 수 불일치: test.csv={len(test_df)}, sample_submission={len(sample_df)}"
        )


def validate_id_alignment(test_df: pd.DataFrame, sample_df: pd.DataFrame, id_col: str):
    if "idx" in test_df.columns:
        left = test_df["idx"].astype(str).reset_index(drop=True)
        right = sample_df[id_col].astype(str).reset_index(drop=True)

        if not left.equals(right):
            mismatches = int((left != right).sum())
            raise ValueError(
                f"test.csv idx와 sample submission의 {id_col}가 일치하지 않습니다. "
                f"불일치 개수: {mismatches}"
            )


def prepare_inference_bundle(config: dict = CONFIG) -> dict:
    set_seed(config["seed"])

    test_df = pd.read_csv(config["test_path"])
    sample_df = pd.read_csv(config["sample_submission_path"])

    validate_inputs(test_df, sample_df)
    id_col, target_col = detect_submission_columns(sample_df)
    validate_id_alignment(test_df, sample_df, id_col)

    inference_texts = build_inference_texts(test_df["conversation"])

    return {
        "test_df": test_df,
        "sample_df": sample_df,
        "id_col": id_col,
        "target_col": target_col,
        "inference_texts": inference_texts,
    }


# ============================================================
# 3. 모델/체크포인트 로드
#    - checkpoint 폴더 직접 지정 가능
#    - results_dktc_xxx 같은 상위 폴더 지정 시 내부 checkpoint-* 자동 탐색
# ============================================================
def _checkpoint_step(path: Path) -> int:
    match = re.search(r"checkpoint-(\d+)$", path.name)
    return int(match.group(1)) if match else -1


def resolve_model_dir(model_path: str | Path) -> Path:
    path = Path(model_path)

    if not path.exists():
        raise FileNotFoundError(f"모델 경로가 없습니다: {path}")

    # 1) 이미 checkpoint 폴더 또는 저장된 HF 모델 폴더인 경우
    if (path / "config.json").exists():
        return path

    # 2) 상위 실험 폴더인 경우 내부 checkpoint-* 자동 선택
    ckpt_dirs = [p for p in path.glob("checkpoint-*") if p.is_dir() and (p / "config.json").exists()]
    ckpt_dirs = sorted(ckpt_dirs, key=_checkpoint_step)

    if ckpt_dirs:
        return ckpt_dirs[-1]

    raise FileNotFoundError(
        f"유효한 checkpoint를 찾지 못했습니다: {path}\n"
        "직접 checkpoint 폴더를 지정하거나, checkpoint-*가 들어 있는 상위 폴더를 지정하세요."
    )


def load_model_bundle(model_path: str | Path, device: str):
    resolved_path = resolve_model_dir(model_path)

    tokenizer = AutoTokenizer.from_pretrained(str(resolved_path), use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(resolved_path))

    model_num_labels = getattr(model.config, "num_labels", None)
    if model_num_labels != NUM_LABELS:
        raise ValueError(
            f"모델 num_labels={model_num_labels} 입니다. "
            f"이 과제는 {NUM_LABELS}개 클래스 모델이어야 합니다."
        )

    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            raise ValueError("tokenizer.pad_token이 없고 대체 가능한 eos_token도 없습니다.")

    added_vocab = tokenizer.get_added_vocab()
    full_vocab = tokenizer.get_vocab()
    if TURN_TOKEN not in added_vocab and TURN_TOKEN not in full_vocab:
        raise ValueError(
            f"tokenizer에 {TURN_TOKEN} special token이 없습니다. "
            "학습에 사용한 checkpoint/tokenizer 저장본인지 확인하세요."
        )

    model.to(device)
    model.eval()
    return tokenizer, model, resolved_path


@torch.no_grad()
def predict_logits(
    texts,
    tokenizer,
    model,
    device,
    batch_size: int = 32,
    max_length: int = 512,
):
    logits_all = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits.detach().cpu().numpy().astype(np.float32)
        logits_all.append(logits)

    if not logits_all:
        raise ValueError("예측할 텍스트가 없습니다.")

    logits_all = np.vstack(logits_all)

    if logits_all.shape != (len(texts), NUM_LABELS):
        raise ValueError(
            f"logits shape 이상: {logits_all.shape}, 기대값=({len(texts)}, {NUM_LABELS})"
        )

    return logits_all


def logits_to_preds(logits: np.ndarray) -> np.ndarray:
    preds = np.argmax(logits, axis=1).astype(int)

    if len(preds) != len(logits):
        raise ValueError(f"예측 개수 불일치: preds={len(preds)}, logits={len(logits)}")

    if not np.isin(preds, np.arange(NUM_LABELS)).all():
        bad_values = sorted(set(preds.tolist()) - set(range(NUM_LABELS)))
        raise ValueError(f"허용 범위(0~{NUM_LABELS-1}) 밖 라벨이 있습니다: {bad_values}")

    return preds


def save_submission(sample_df: pd.DataFrame, target_col: str, preds: np.ndarray, output_name: str, config: dict = CONFIG):
    if not str(output_name).lower().endswith(".csv"):
        output_name = f"{output_name}.csv"

    submit_df = sample_df.copy()
    submit_df[target_col] = preds.astype(int)

    config["output_dir"].mkdir(parents=True, exist_ok=True)
    output_path = config["output_dir"] / output_name
    submit_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("=" * 70)
    print(f"output_path: {output_path}")
    print(f"rows       : {len(submit_df)}")
    print(f"columns    : {submit_df.columns.tolist()}")
    print(f"label_dist : {submit_df[target_col].value_counts().sort_index().to_dict()}")
    print("=" * 70)

    return submit_df, output_path


def _cleanup_model_objects(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 4. 제출 생성
# ============================================================
def make_submission(model_path: str | Path, output_name: str, config: dict = CONFIG):
    """단일 모델 제출 파일 생성"""
    bundle = prepare_inference_bundle(config)

    tokenizer, model, resolved_path = load_model_bundle(model_path, config["device"])
    logits = predict_logits(
        texts=bundle["inference_texts"],
        tokenizer=tokenizer,
        model=model,
        device=config["device"],
        batch_size=config["batch_size"],
        max_length=config["max_length"],
    )
    preds = logits_to_preds(logits)
    submit_df, output_path = save_submission(
        sample_df=bundle["sample_df"],
        target_col=bundle["target_col"],
        preds=preds,
        output_name=output_name,
        config=config,
    )

    print(f"model_path : {resolved_path}")
    print(f"device     : {config['device']}")
    print(f"sample_text: {bundle['inference_texts'][0][:120] if bundle['inference_texts'] else 'N/A'}")

    _cleanup_model_objects(model, tokenizer)
    return submit_df, logits, output_path


def make_multiple_submissions(job_list, config: dict = CONFIG):
    """단일 모델 여러 개 제출"""
    bundle = prepare_inference_bundle(config)
    results = {}

    for job in job_list:
        if "model_path" not in job or "output_name" not in job:
            raise ValueError(f"job 항목에 model_path 또는 output_name이 없습니다: {job}")

        tokenizer, model, resolved_path = load_model_bundle(job["model_path"], config["device"])
        logits = predict_logits(
            texts=bundle["inference_texts"],
            tokenizer=tokenizer,
            model=model,
            device=config["device"],
            batch_size=config["batch_size"],
            max_length=config["max_length"],
        )
        preds = logits_to_preds(logits)
        submit_df, output_path = save_submission(
            sample_df=bundle["sample_df"],
            target_col=bundle["target_col"],
            preds=preds,
            output_name=job["output_name"],
            config=config,
        )

        print(f"model_path : {resolved_path}")
        print(f"device     : {config['device']}")
        results[job["output_name"]] = {
            "df": submit_df,
            "logits": logits,
            "output_path": output_path,
            "resolved_model_path": str(resolved_path),
        }
        _cleanup_model_objects(model, tokenizer)

    return results


def make_ensemble_submission(
    model_paths,
    output_name: str,
    weights=None,
    config: dict = CONFIG,
):
    """
    여러 모델의 logits를 가중 평균한 뒤 제출 파일 생성
    - model_paths: checkpoint 폴더 또는 상위 실험 폴더 목록
    - weights    : None이면 동일 가중치
    """
    if not model_paths:
        raise ValueError("model_paths가 비어 있습니다.")

    if weights is None:
        weights = [1.0] * len(model_paths)

    weights = np.asarray(weights, dtype=np.float64)
    if weights.shape[0] != len(model_paths):
        raise ValueError(f"weights 길이={len(weights)} / model_paths 길이={len(model_paths)} 불일치")
    if np.any(weights < 0):
        raise ValueError("weights에는 음수가 들어갈 수 없습니다.")
    if float(weights.sum()) <= 0:
        raise ValueError("weights 합계가 0보다 커야 합니다.")

    bundle = prepare_inference_bundle(config)

    ensemble_logits = None
    resolved_paths = []

    for i, (model_path, weight) in enumerate(zip(model_paths, weights), start=1):
        tokenizer, model, resolved_path = load_model_bundle(model_path, config["device"])
        logits = predict_logits(
            texts=bundle["inference_texts"],
            tokenizer=tokenizer,
            model=model,
            device=config["device"],
            batch_size=config["batch_size"],
            max_length=config["max_length"],
        ).astype(np.float64)

        if ensemble_logits is None:
            ensemble_logits = weight * logits
        else:
            if ensemble_logits.shape != logits.shape:
                raise ValueError(
                    f"logits shape 불일치: ensemble={ensemble_logits.shape}, current={logits.shape}"
                )
            ensemble_logits += weight * logits

        resolved_paths.append(str(resolved_path))
        print(f"[{i}/{len(model_paths)}] loaded: {resolved_path} | weight={float(weight):.4f}")
        _cleanup_model_objects(model, tokenizer)

    ensemble_logits /= float(weights.sum())
    preds = logits_to_preds(ensemble_logits)

    submit_df, output_path = save_submission(
        sample_df=bundle["sample_df"],
        target_col=bundle["target_col"],
        preds=preds,
        output_name=output_name,
        config=config,
    )

    print(f"ensemble_size : {len(model_paths)}")
    print(f"resolved_paths: {resolved_paths}")
    print(f"weights       : {weights.tolist()}")
    print(f"device        : {config['device']}")
    print(f"sample_text   : {bundle['inference_texts'][0][:120] if bundle['inference_texts'] else 'N/A'}")

    return submit_df, ensemble_logits, output_path, resolved_paths


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 5. 제출 실행
#    권장: K-Fold 5개 모델 평균 logit 제출
#    - roberta-base_hp_Focal_Loss_v2.ipynb 기준 output_dir:
#      ./results_dktc_kfold_1of5
#      ./results_dktc_kfold_2of5
#      ./results_dktc_kfold_3of5
#      ./results_dktc_kfold_4of5
#      ./results_dktc_kfold_5of5
# ============================================================

kfold_model_paths = [
    "./results_dktc_kfold_1of5",
    "./results_dktc_kfold_2of5",
    "./results_dktc_kfold_3of5",
    "./results_dktc_kfold_4of5",
    "./results_dktc_kfold_5of5",
]

kfold_submit_df, kfold_logits, kfold_output_path, kfold_resolved_paths = make_ensemble_submission(
    model_paths=kfold_model_paths,
    output_name="sub_kfold_5fold_avglogits_v3.csv",
)


# ------------------------------------------------------------
# 옵션 1) 상위 단일 모델 제출
# ------------------------------------------------------------
# single_jobs = [
#     {"model_path": "./results_dktc_wd_0.1",      "output_name": "sub_wd_0.1_v3.csv"},
#     {"model_path": "./results_dktc_no_weight",   "output_name": "sub_no_weight_v3.csv"},
#     {"model_path": "./results_dktc_focal_g2.0",  "output_name": "sub_focal_g2.0_v3.csv"},
# ]
# single_results = make_multiple_submissions(single_jobs)


# ------------------------------------------------------------
# 옵션 2) 상위 3개 단일 모델 평균 logit 앙상블
# ------------------------------------------------------------
# top3_model_paths = [
#     "./results_dktc_wd_0.1",
#     "./results_dktc_no_weight",
#     "./results_dktc_focal_g2.0",
# ]
#
# top3_submit_df, top3_logits, top3_output_path, top3_resolved_paths = make_ensemble_submission(
#     model_paths=top3_model_paths,
#     output_name="sub_top3_avglogits_v3.csv",
# )


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1354.21it/s]


[1/5] loaded: results_dktc_kfold_1of5/checkpoint-738 | weight=1.0000


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1612.91it/s]


[2/5] loaded: results_dktc_kfold_2of5/checkpoint-738 | weight=1.0000


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1776.54it/s]


[3/5] loaded: results_dktc_kfold_3of5/checkpoint-738 | weight=1.0000


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1695.75it/s]


[4/5] loaded: results_dktc_kfold_4of5/checkpoint-738 | weight=1.0000


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1610.13it/s]


[5/5] loaded: results_dktc_kfold_5of5/checkpoint-738 | weight=1.0000
output_path: submission/sub_kfold_5fold_avglogits_v3.csv
rows       : 500
columns    : ['file_name', 'class']
label_dist : {0: 110, 1: 102, 2: 111, 3: 134, 4: 43}
ensemble_size : 5
resolved_paths: ['results_dktc_kfold_1of5/checkpoint-738', 'results_dktc_kfold_2of5/checkpoint-738', 'results_dktc_kfold_3of5/checkpoint-738', 'results_dktc_kfold_4of5/checkpoint-738', 'results_dktc_kfold_5of5/checkpoint-738']
weights       : [1.0, 1.0, 1.0, 1.0, 1.0]
device        : cuda
sample_text   : 아가씨 담배한갑주소 네 4500원입니다 어 네 지갑어디갔지 에이 버스에서 잃어버렸나보네 그럼 취소할까요 아가씨 내 여기단골이니 담에 갖다줄께 저도 알바생이라 외상안됩니다 아따 누가 떼먹는다고 그러나 갖다준다고 안됩니
